In [1]:
import pandas as pd

utci = pd.read_csv(
    "../data/processed/utci_hourly.csv",
    nrows=5
)

print("Columns:")
print(utci.columns.tolist())

print("\nData:")
display(utci)

Columns:
['time', 'latitude', 'longitude', 'tdb', 'mrt', 'v', 'rh', 'utci', 'number', 'step', 'surface', 'valid_time']

Data:


,time,latitude,longitude,tdb,mrt,v,rh,utci,number,step,surface,valid_time
0,2021-04-01 00:00:00,23.25,72.25,22.828949,8.785309,2.499773,50.233260,16.289580,0,0 days,0.0,2021-04-01 00:00:00
1,2021-04-01 00:00:00,23.25,72.50,22.635590,8.334778,2.416389,41.109910,15.608298,0,0 days,0.0,2021-04-01 00:00:00
2,2021-04-01 00:00:00,23.25,72.75,22.196136,7.868805,2.510747,44.236458,15.081756,0,0 days,0.0,2021-04-01 00:00:00
3,2021-04-01 00:00:00,23.00,72.25,24.106293,10.269684,2.510639,64.983520,18.947195,0,0 days,0.0,2021-04-01 00:00:00
4,2021-04-01 00:00:00,23.00,72.50,23.869965,9.863403,2.418709,55.996450,18.089115,0,0 days,0.0,2021-04-01 00:00:00


In [2]:
import geopandas as gpd

wards = gpd.read_file(
    "../data/spatial/ahmedabad_wards_clean.geojson"
)

print("Number of wards:", len(wards))
print("CRS:", wards.crs)

display(wards[["Ward_ID", "Ward_Name"]].head())

Number of wards: 48
CRS: EPSG:4326


,Ward_ID,Ward_Name
0,1,Ramol Hathijan
1,2,Vatva
2,3,Lambha
3,4,Isanpur
4,5,Khokhra


In [3]:
# Convert UTCI latitude/longitude into spatial points

utci_points = gpd.GeoDataFrame(
    utci,
    geometry=gpd.points_from_xy(
        utci["longitude"],
        utci["latitude"]
    ),
    crs="EPSG:4326"
)

print("Number of UTCI points:", len(utci_points))
print("CRS:", utci_points.crs)

display(
    utci_points[
        ["time", "latitude", "longitude", "utci", "geometry"]
    ]
)

Number of UTCI points: 5
CRS: EPSG:4326


,time,latitude,longitude,utci,geometry
0,2021-04-01 00:00:00,23.25,72.25,16.289580,POINT (72.25 23.25)
1,2021-04-01 00:00:00,23.25,72.50,15.608298,POINT (72.5 23.25)
2,2021-04-01 00:00:00,23.25,72.75,15.081756,POINT (72.75 23.25)
3,2021-04-01 00:00:00,23.00,72.25,18.947195,POINT (72.25 23)
4,2021-04-01 00:00:00,23.00,72.50,18.089115,POINT (72.5 23)


In [4]:
# Test spatial join: assign each UTCI point to its Ahmedabad ward

test_join = gpd.sjoin(
    utci_points,
    wards[["Ward_ID", "Ward_Name", "geometry"]],
    how="left",
    predicate="within"
)

print("Spatial join result:")
display(
    test_join[
        ["time", "latitude", "longitude", "utci", "Ward_ID", "Ward_Name"]
    ]
)

Spatial join result:


,time,latitude,longitude,utci,Ward_ID,Ward_Name
0,2021-04-01 00:00:00,23.25,72.25,16.289580,NaN,NaN
1,2021-04-01 00:00:00,23.25,72.50,15.608298,NaN,NaN
2,2021-04-01 00:00:00,23.25,72.75,15.081756,NaN,NaN
3,2021-04-01 00:00:00,23.00,72.25,18.947195,NaN,NaN
4,2021-04-01 00:00:00,23.00,72.50,18.089115,16.0,Sarkhej


In [5]:
# Load the complete UTCI dataset

utci = pd.read_csv(
    "../data/processed/utci_hourly.csv"
)

print("Rows:", len(utci))
print("Columns:", utci.columns.tolist())

print("\nUnique latitude values:", utci["latitude"].nunique())
print("Unique longitude values:", utci["longitude"].nunique())

print("\nDate range:")
print(utci["time"].min(), "to", utci["time"].max())

Rows: 117936
Columns: ['time', 'latitude', 'longitude', 'tdb', 'mrt', 'v', 'rh', 'utci', 'number', 'step', 'surface', 'valid_time']

Unique latitude values: 3
Unique longitude values: 3

Date range:
2021-04-01 00:00:00 to 2026-06-30 23:00:00


In [6]:
grid_points = (
    utci[["latitude", "longitude"]]
    .drop_duplicates()
    .sort_values(["latitude", "longitude"])
    .reset_index(drop=True)
)

print("Number of unique ERA5 grid points:", len(grid_points))
display(grid_points)

Number of unique ERA5 grid points: 9


,latitude,longitude
0,22.75,72.25
1,22.75,72.50
2,22.75,72.75
3,23.00,72.25
4,23.00,72.50
5,23.00,72.75
6,23.25,72.25
7,23.25,72.50
8,23.25,72.75


In [7]:
# Create a centroid for each Ahmedabad ward

# Use a projected CRS for accurate centroid calculation
wards_projected = wards.to_crs("EPSG:32643")

wards_centroids = wards_projected.copy()
wards_centroids["geometry"] = wards_projected.geometry.centroid

# Convert centroids back to latitude/longitude
wards_centroids = wards_centroids.to_crs("EPSG:4326")

print("Number of ward centroids:", len(wards_centroids))

display(
    wards_centroids[
        ["Ward_ID", "Ward_Name", "geometry"]
    ].head()
)

Number of ward centroids: 48


,Ward_ID,Ward_Name,geometry
0,1,Ramol Hathijan,POINT (72.6482 22.95764)
1,2,Vatva,POINT (72.61781 22.94975)
2,3,Lambha,POINT (72.56492 22.95002)
3,4,Isanpur,POINT (72.60003 22.97977)
4,5,Khokhra,POINT (72.61593 22.99566)


In [8]:
# Create GeoDataFrame from the 9 unique ERA5 grid points

era5_grid = gpd.GeoDataFrame(
    grid_points,
    geometry=gpd.points_from_xy(
        grid_points["longitude"],
        grid_points["latitude"]
    ),
    crs="EPSG:4326"
)

print("ERA5 grid points:", len(era5_grid))
display(era5_grid)

ERA5 grid points: 9


,latitude,longitude,geometry
0,22.75,72.25,POINT (72.25 22.75)
1,22.75,72.50,POINT (72.5 22.75)
2,22.75,72.75,POINT (72.75 22.75)
3,23.00,72.25,POINT (72.25 23)
4,23.00,72.50,POINT (72.5 23)
5,23.00,72.75,POINT (72.75 23)
6,23.25,72.25,POINT (72.25 23.25)
7,23.25,72.50,POINT (72.5 23.25)
8,23.25,72.75,POINT (72.75 23.25)


In [9]:
# Assign the nearest ERA5 grid point to each ward centroid

ward_to_grid = gpd.sjoin_nearest(
    wards_centroids,
    era5_grid,
    how="left",
    distance_col="distance"
)

print("Rows:", len(ward_to_grid))

display(
    ward_to_grid[
        ["Ward_ID", "Ward_Name",
         "latitude", "longitude", "distance"]
    ].head(10)
)

Rows: 48


C:\Users\91965\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\geopandas\array.py:411: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


,Ward_ID,Ward_Name,latitude,longitude,distance
0,1,Ramol Hathijan,23.0,72.75,0.110261
1,2,Vatva,23.0,72.50,0.128078
2,3,Lambha,23.0,72.50,0.081926
3,4,Isanpur,23.0,72.50,0.102053
4,5,Khokhra,23.0,72.50,0.116012
5,6,Bhaipura Hatkeshwar,23.0,72.75,0.122236
6,7,Indrapuri,23.0,72.75,0.111037
7,8,Vastral,23.0,72.75,0.081261
8,9,Odhav,23.0,72.75,0.091817
9,10,Amraiwadi,23.0,72.50,0.124505


In [10]:
# Keep only the information needed to connect UTCI to wards

ward_grid_lookup = ward_to_grid[
    ["Ward_ID", "Ward_Name", "latitude", "longitude"]
].copy()

print("Ward-grid lookup:")
display(ward_grid_lookup.head())

Ward-grid lookup:


,Ward_ID,Ward_Name,latitude,longitude
0,1,Ramol Hathijan,23.0,72.75
1,2,Vatva,23.0,72.50
2,3,Lambha,23.0,72.50
3,4,Isanpur,23.0,72.50
4,5,Khokhra,23.0,72.50


In [11]:
# Connect every hourly UTCI observation to its assigned ward(s)

utci_ward = utci.merge(
    ward_grid_lookup,
    on=["latitude", "longitude"],
    how="inner"
)

print("Rows after ward mapping:", len(utci_ward))

print("\nColumns:")
print(utci_ward.columns.tolist())

display(
    utci_ward[
        ["time", "latitude", "longitude", "utci",
         "Ward_ID", "Ward_Name"]
    ].head(10)
)

Rows after ward mapping: 628992

Columns:
['time', 'latitude', 'longitude', 'tdb', 'mrt', 'v', 'rh', 'utci', 'number', 'step', 'surface', 'valid_time', 'Ward_ID', 'Ward_Name']


,time,latitude,longitude,utci,Ward_ID,Ward_Name
0,2021-04-01 00:00:00,23.0,72.5,18.089115,2,Vatva
1,2021-04-01 00:00:00,23.0,72.5,18.089115,3,Lambha
2,2021-04-01 00:00:00,23.0,72.5,18.089115,4,Isanpur
3,2021-04-01 00:00:00,23.0,72.5,18.089115,5,Khokhra
4,2021-04-01 00:00:00,23.0,72.5,18.089115,10,Amraiwadi
5,2021-04-01 00:00:00,23.0,72.5,18.089115,11,Gomtipur
6,2021-04-01 00:00:00,23.0,72.5,18.089115,12,Maninagar
7,2021-04-01 00:00:00,23.0,72.5,18.089115,13,Danilimda
8,2021-04-01 00:00:00,23.0,72.5,18.089115,14,Baherampura
9,2021-04-01 00:00:00,23.0,72.5,18.089115,15,Maktampura


In [12]:
# Convert time to date
utci_ward["time"] = pd.to_datetime(utci_ward["time"])
utci_ward["date"] = utci_ward["time"].dt.date

# Calculate daily UTCI statistics for each ward
ward_daily_utci = (
    utci_ward
    .groupby(["Ward_ID", "Ward_Name", "date"], as_index=False)
    .agg(
        UTCI_mean=("utci", "mean"),
        UTCI_max=("utci", "max"),
        UTCI_min=("utci", "min")
    )
)

print("Rows:", len(ward_daily_utci))
print("Number of wards:", ward_daily_utci["Ward_ID"].nunique())
print("Date range:",
      ward_daily_utci["date"].min(),
      "to",
      ward_daily_utci["date"].max())

display(ward_daily_utci.head(10))

Rows: 26208
Number of wards: 48
Date range: 2021-04-01 to 2026-06-30


,Ward_ID,Ward_Name,date,UTCI_mean,UTCI_max,UTCI_min
0,1,Ramol Hathijan,2021-04-01,28.865797,43.229015,16.560957
1,1,Ramol Hathijan,2021-04-02,30.250199,43.855992,20.187847
2,1,Ramol Hathijan,2021-04-03,31.136628,44.630685,20.768254
3,1,Ramol Hathijan,2021-04-04,31.611155,44.593984,21.994999
4,1,Ramol Hathijan,2021-04-05,32.334757,44.728843,20.580646
5,1,Ramol Hathijan,2021-04-06,31.780229,44.621734,20.537270
6,1,Ramol Hathijan,2021-04-07,31.489122,44.514046,20.535202
7,1,Ramol Hathijan,2021-04-08,31.032477,44.227353,17.706249
8,1,Ramol Hathijan,2021-04-09,30.011143,44.502454,18.158775
9,1,Ramol Hathijan,2021-04-10,30.485894,44.803534,17.287345


In [13]:
# Load existing CTBI/NCTL results

ctbi = pd.read_csv("../data/processed/CTBI.csv")

print("CTBI rows:", len(ctbi))
print("CTBI columns:", ctbi.columns.tolist())

# Standardize date
ctbi["date"] = pd.to_datetime(ctbi["date"]).dt.date

print("\nCTBI date range:")
print(ctbi["date"].min(), "to", ctbi["date"].max())

display(ctbi.head())

CTBI rows: 546
CTBI columns: ['date', 'DayStress', 'year', 'NCTL', 'DayStress_norm', 'NCTL_norm', 'CTBI']

CTBI date range:
2021-04-01 to 2026-06-30


,date,DayStress,year,NCTL,DayStress_norm,NCTL_norm,CTBI
0,2021-04-01,37.309403,2021,0.0,0.684861,0.0,0.102729
1,2021-04-02,38.303439,2021,0.0,0.726934,0.0,0.180950
2,2021-04-03,38.912634,2021,0.0,0.752718,0.0,0.239573
3,2021-04-04,39.233238,2021,0.0,0.766288,0.0,0.282644
4,2021-04-05,39.910378,2021,0.0,0.794948,0.0,0.317093


In [14]:
# Make sure ward dates have the same format
ward_daily_utci["date"] = pd.to_datetime(
    ward_daily_utci["date"]
).dt.date

# Keep the CTBI variables needed for integration
ctbi_daily = ctbi[
    ["date", "NCTL", "DayStress", "DayStress_norm", "NCTL_norm", "CTBI"]
].copy()

# Merge thermal indicators with ward-level UTCI
ward_thermal = ward_daily_utci.merge(
    ctbi_daily,
    on="date",
    how="left"
)

print("Rows:", len(ward_thermal))
print("Number of wards:", ward_thermal["Ward_ID"].nunique())

print("\nMissing values:")
print(
    ward_thermal[
        ["UTCI_mean", "UTCI_max", "UTCI_min",
         "NCTL", "DayStress", "CTBI"]
    ].isna().sum()
)

display(ward_thermal.head())

Rows: 26208
Number of wards: 48

Missing values:
UTCI_mean    0
UTCI_max     0
UTCI_min     0
NCTL         0
DayStress    0
CTBI         0
dtype: int64


,Ward_ID,Ward_Name,date,UTCI_mean,UTCI_max,UTCI_min,NCTL,DayStress,DayStress_norm,NCTL_norm,CTBI
0,1,Ramol Hathijan,2021-04-01,28.865797,43.229015,16.560957,0.0,37.309403,0.684861,0.0,0.102729
1,1,Ramol Hathijan,2021-04-02,30.250199,43.855992,20.187847,0.0,38.303439,0.726934,0.0,0.180950
2,1,Ramol Hathijan,2021-04-03,31.136628,44.630685,20.768254,0.0,38.912634,0.752718,0.0,0.239573
3,1,Ramol Hathijan,2021-04-04,31.611155,44.593984,21.994999,0.0,39.233238,0.766288,0.0,0.282644
4,1,Ramol Hathijan,2021-04-05,32.334757,44.728843,20.580646,0.0,39.910378,0.794948,0.0,0.317093


In [15]:
# Validate ward coverage

print("Total wards:", ward_thermal["Ward_ID"].nunique())

missing_wards = set(wards["Ward_ID"]) - set(
    ward_thermal["Ward_ID"]
)

print("Missing Ward IDs:", missing_wards)

print("\nRecords per ward:")
print(
    ward_thermal.groupby("Ward_ID").size().describe()
)

Total wards: 48
Missing Ward IDs: set()

Records per ward:
count     48.0
mean     546.0
std        0.0
min      546.0
25%      546.0
50%      546.0
75%      546.0
max      546.0
dtype: float64


In [16]:
# Load demographic join template

demographic = pd.read_csv(
    "../data/spatial/demographic_join_template.csv"
)

print("Rows:", len(demographic))
print("Columns:")
print(demographic.columns.tolist())

display(demographic.head(10))

Rows: 48
Columns:
['Ward_ID', 'Ward_Name', 'population_total', 'population_density', 'elderly_pct', 'sc_st_pct', 'literacy_pct', 'outdoor_worker_density', 'slum_density', 'healthcare_access', 'ndvi_green_cover']


,Ward_ID,Ward_Name,population_total,population_density,elderly_pct,sc_st_pct,literacy_pct,outdoor_worker_density,slum_density,healthcare_access,ndvi_green_cover
0,1,Ramol Hathijan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Vatva,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Lambha,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,Isanpur,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Khokhra,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,6,Bhaipura Hatkeshwar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,7,Indrapuri,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,8,Vastral,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,9,Odhav,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,10,Amraiwadi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
print("Rows:", len(ward_thermal))
print("Wards:", ward_thermal["Ward_ID"].nunique())
print("Dates:", ward_thermal["date"].nunique())

display(ward_thermal.head())

Rows: 26208
Wards: 48
Dates: 546


,Ward_ID,Ward_Name,date,UTCI_mean,UTCI_max,UTCI_min,NCTL,DayStress,DayStress_norm,NCTL_norm,CTBI
0,1,Ramol Hathijan,2021-04-01,28.865797,43.229015,16.560957,0.0,37.309403,0.684861,0.0,0.102729
1,1,Ramol Hathijan,2021-04-02,30.250199,43.855992,20.187847,0.0,38.303439,0.726934,0.0,0.180950
2,1,Ramol Hathijan,2021-04-03,31.136628,44.630685,20.768254,0.0,38.912634,0.752718,0.0,0.239573
3,1,Ramol Hathijan,2021-04-04,31.611155,44.593984,21.994999,0.0,39.233238,0.766288,0.0,0.282644
4,1,Ramol Hathijan,2021-04-05,32.334757,44.728843,20.580646,0.0,39.910378,0.794948,0.0,0.317093


In [18]:
ward_final_geo = wards[
    ["Ward_ID", "Ward_Name", "geometry"]
].merge(
    ward_thermal,
    on=["Ward_ID", "Ward_Name"],
    how="inner"
)

ward_final_geo = gpd.GeoDataFrame(
    ward_final_geo,
    geometry="geometry",
    crs=wards.crs
)

print("Final rows:", len(ward_final_geo))
print("Final wards:", ward_final_geo["Ward_ID"].nunique())

print("Invalid geometries:",
      (~ward_final_geo.geometry.is_valid).sum())

print("Missing geometry:",
      ward_final_geo.geometry.isna().sum())

Final rows: 26208
Final wards: 48
Invalid geometries: 0
Missing geometry: 0


In [19]:
ward_final_geo.to_csv(
    "../data/processed/ward_thermal_daily.csv",
    index=False
)

ward_final_geo.to_file(
    "../data/processed/ward_thermal_daily.geojson",
    driver="GeoJSON"
)

print("Day 3 spatial files saved successfully.")

Day 3 spatial files saved successfully.
